In [ ]:
import pandas as pd
import numpy as np


def add_date_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert the date column to datetime and derive Year, Month, and
    Day-of-Year (DOY) features used for seasonality.

    Args:
        df (pd.DataFrame): Raw merged dataframe with a 'date' column.

    Returns:
        pd.DataFrame: Copy of df with 'date' converted to datetime and
        'Year', 'Month', 'DOY' columns added.
    """
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df["Year"] = df["date"].dt.year
    df["Month"] = df["date"].dt.month
    df["DOY"] = df["date"].dt.dayofyear
    return df


def add_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add lag and rolling-window features for rainfall and key weather
    variables, computed per station. Invalidates (sets to NaN) any
    lag/rolling value that spans a real gap in a station's date
    record, since a naive shift() would otherwise silently pull a
    value from before a multi-year gap.

    Args:
        df (pd.DataFrame): Dataframe with 'Station', 'date',
        'BMD_Rainfall', 'T2M', 'RH2M', 'PS' columns, sorted or
        unsorted (function sorts internally).

    Returns:
        pd.DataFrame: Copy of df with Rain_lag1, Rain_lag2,
        Rain_7day_avg, Temp_lag1, Humidity_lag1, Pressure_lag1 added,
        nulled across station date-gaps.
    """
    df = df.sort_values(["Station", "date"]).copy()

    df["prev_date"] = df.groupby("Station")["date"].shift(1)
    df["days_since_prev"] = (df["date"] - df["prev_date"]).dt.days

    df["Rain_lag1"] = df.groupby("Station")["BMD_Rainfall"].shift(1)
    df["Rain_lag2"] = df.groupby("Station")["BMD_Rainfall"].shift(2)
    df["Rain_7day_avg"] = (
        df.groupby("Station")["BMD_Rainfall"]
        .transform(lambda x: x.shift(1).rolling(window=7, min_periods=7).mean())
    )
    df["Temp_lag1"] = df.groupby("Station")["T2M"].shift(1)
    df["Humidity_lag1"] = df.groupby("Station")["RH2M"].shift(1)
    df["Pressure_lag1"] = df.groupby("Station")["PS"].shift(1)

    gap_mask = df["days_since_prev"] != 1
    lag_cols = ["Rain_lag1", "Rain_lag2", "Rain_7day_avg",
                "Temp_lag1", "Humidity_lag1", "Pressure_lag1"]
    for col in lag_cols:
        df.loc[gap_mask, col] = np.nan

    df["days_since_prev2"] = df.groupby("Station")["days_since_prev"].shift(1)
    df.loc[df["days_since_prev2"] != 1, "Rain_lag2"] = np.nan

    return df.drop(columns=["prev_date", "days_since_prev", "days_since_prev2"])


def add_station_target_encoding(df: pd.DataFrame, train_end_year: int = 2007) -> tuple[pd.DataFrame, pd.Series]:
    """
    Encode the Station column using target encoding: each station is
    replaced with its mean historical rainfall, computed using only
    training-period years to avoid leaking future information into a
    feature seen by the model during earlier training years.

    Target encoding is used instead of One-Hot Encoding because with
    34 stations feeding one global model, it lets the model learn a
    continuous "this station tends to be wet/dry" signal rather than
    34 independent binary flags with no shared information between
    similar stations.

    Args:
        df (pd.DataFrame): Dataframe with 'Station', 'Year', and
        'BMD_Rainfall' columns.
        train_end_year (int): Last year (inclusive) considered part
        of the training period. Defaults to 2007, matching the
        project's locked chronological split (train: 1984-2007).

    Returns:
        tuple[pd.DataFrame, pd.Series]: df with a new
        'Station_TargetEnc' column, and the fitted station-mean
        mapping (must be saved and reused at inference time).
    """
    df = df.copy()
    train_mask = df["Year"] <= train_end_year

    station_means = df.loc[train_mask].groupby("Station")["BMD_Rainfall"].mean()
    global_mean = df.loc[train_mask, "BMD_Rainfall"].mean()

    df["Station_TargetEnc"] = df["Station"].map(station_means).fillna(global_mean)
    return df, station_means


def flag_outliers_iqr(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """
    Detect outliers in the given columns using the IQR method and
    return a summary of flagged counts and bounds. Does NOT remove
    or modify any values - flagging only, since this project's
    weather/rainfall extremes largely reflect genuine meteorological
    events (e.g., monsoon rainfall, hill-station pressure) rather
    than data entry errors.

    Args:
        df (pd.DataFrame): Dataframe to check.
        columns (list[str]): Numeric column names to check for
        outliers. Should exclude the rainfall target column, whose
        extremes are the signal being modeled, not noise to remove.

    Returns:
        pd.DataFrame: One row per column with flagged count,
        percentage, and IQR bounds.
    """
    results = []
    for col in columns:
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        mask = (df[col] < lower) | (df[col] > upper)
        results.append({
            "column": col,
            "flagged_count": int(mask.sum()),
            "flagged_pct": round(mask.sum() / len(df) * 100, 3),
            "lower_bound": round(lower, 3),
            "upper_bound": round(upper, 3),
        })
    return pd.DataFrame(results)